In [5]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pays = {
    'Benin':        (9.30, 2.32),
    'Senegal':      (14.49, -14.45),
    'Ghana':        (7.95, -1.02),
    'Nigeria':      (9.08, 8.67),
    'Mali':         (17.57, -3.99),
    'Burkina Faso': (12.36, -1.53),
    'Cote dIvoire': (7.54, -5.55),
}

retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
    raise_on_status=False,
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))
session.mount("http://", HTTPAdapter(max_retries=retry_strategy))

all_data = []

for country, (lat, lon) in pays.items():
    print(f"Collecte {country}...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": "2000-01-01", "end_date": "2023-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "auto"
    }

    response = session.get(url, params=params, timeout=60)
    if response.status_code != 200:
        print(f"Réponse HTTP inattendue pour {country} : {response.status_code}")
        print(response.text)
        continue

    data = response.json()
    if "daily" not in data:
        print(f"Réponse inattendue pour {country} : {data}")
        continue

    df = pd.DataFrame(data["daily"])
    df["date"] = pd.to_datetime(df["time"] )
    df["year"] = df["date"].dt.year
    df["country"] = country
    yearly = df.groupby(["country", "year"]).agg(
        avg_temp_c=("temperature_2m_mean", "mean"),
        total_precip_mm=("precipitation_sum", "sum")
    ).reset_index()
    all_data.append(yearly)

if not all_data:
    raise RuntimeError("Aucune donnée météo n'a pu être récupérée.")

meteo_df = pd.concat(all_data, ignore_index=True)
meteo_df.to_csv("meteo_clean.csv", index=False)
print(f"Done ! Shape : {meteo_df.shape}")
print(meteo_df.head(10))

Collecte Benin...
Collecte Senegal...
Collecte Ghana...
Collecte Nigeria...
Collecte Mali...
Collecte Burkina Faso...
Collecte Cote dIvoire...
Done ! Shape : (168, 4)
  country  year  avg_temp_c  total_precip_mm
0   Benin  2000   26.907377           1298.6
1   Benin  2001   26.925205            984.6
2   Benin  2002   27.001096           1332.9
3   Benin  2003   27.223288           1153.4
4   Benin  2004   26.996721            983.2
5   Benin  2005   27.240548           1041.3
6   Benin  2006   27.243288           1222.7
7   Benin  2007   27.011507           1065.8
8   Benin  2008   27.034153           1139.5
9   Benin  2009   27.315068           1156.8
